# Delta Lake tables 
 Use this notebook to explore Delta Lake functionality 

In [32]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# define the schema
schema = StructType() \
.add("ProductID", IntegerType(), True) \
.add("ProductName", StringType(), True) \
.add("Category", StringType(), True) \
.add("ListPrice", DoubleType(), True)

df = spark.read.format("csv").option("header","true").schema(schema).load("Files/products/products.csv")
# df now is a Spark DataFrame containing CSV data from "Files/products/products.csv".
display(df)

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5da68845-da0c-4eee-8d72-2ceb0e69cc3a)

In [33]:
df.write.format("delta").saveAsTable("managed_products")

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 36, Finished, Available, Finished, False)

In [36]:
df.write.format("delta").saveAsTable("external_products", path="abfss://DeltaTables@onelake.dfs.fabric.microsoft.com/lakehouse.Lakehouse/Files/products2")

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 39, Finished, Available, Finished, False)

In [37]:
%%sql
DESCRIBE FORMATTED managed_products;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 40, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 3 fields>

In [38]:
%%sql
DESCRIBE FORMATTED external_products;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 41, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 3 fields>

In [41]:
%%sql
DROP TABLE managed_products;
DROP TABLE external_products;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 45, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [39]:
%%sql
CREATE TABLE products
USING DELTA
LOCATION 'Files/products2';

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 42, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [43]:
%%sql
select * from products where category = "Mountain Bikes"

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 47, Finished, Available, Finished, False)

<Spark SQL result set with 32 rows and 4 fields>

In [45]:
%%sql
UPDATE products
SET ListPrice = ListPrice * 0.9
WHERE Category = 'Mountain Bikes';

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 49, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [47]:
%%sql
DESCRIBE HISTORY products;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 51, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 15 fields>

In [49]:
delta_table_path = 'Files/products2'
# Get the current data
current_data = spark.read.format("delta").load(delta_table_path)
display(current_data)

# Get the version 0 data
original_data = spark.read.format("delta").option("versionAsOf", 1).load(delta_table_path)
display(original_data)

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 90f07c19-abe6-4466-a15c-5e9d79319171)

SynapseWidget(Synapse.DataFrame, 0871145b-c688-4f5a-830b-a6024ba82776)

In [50]:
 %%sql
 SELECT Category, COUNT(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 54, Finished, Available, Finished, False)

<Spark SQL result set with 37 rows and 5 fields>

In [51]:
%%sql
-- Create a temporary view
CREATE OR REPLACE TEMPORARY VIEW products_view
AS
    SELECT Category, COUNT(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category;

SELECT *
FROM products_view
ORDER BY Category; 

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 56, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 37 rows and 5 fields>

In [52]:
%%sql
SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 57, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

In [ ]:
from pyspark.sql.functions import col, desc

df_products = spark.sql("SELECT Category, MinPrice, MaxPrice, AvgPrice FROM products_view").orderBy(col("AvgPrice").desc())
display(df_products.limit(10))

In [55]:
from notebookutils import mssparkutils
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Create a folder
inputPath = 'Files/data/'
mssparkutils.fs.mkdirs(inputPath)

# Create a stream that reads data from the folder, using a JSON schema
jsonSchema = StructType([
StructField("device", StringType(), False),
StructField("status", StringType(), False)
])
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(inputPath)

# Write some event data to the folder
device_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "data.txt", device_data, True)

print("Source stream created...")

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 60, Finished, Available, Finished, False)

Source stream created...


In [56]:
# Write the stream to a delta table
delta_stream_table_path = 'Tables/iotdevicedata'
checkpointpath = 'Files/delta/checkpoint'
deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(delta_stream_table_path)
print("Streaming to delta sink...")

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 61, Finished, Available, Finished, False)

Streaming to delta sink...


In [57]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 62, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 2 fields>

In [60]:
# Add more data to the source stream
more_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "more-data2.txt", more_data, True)

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 65, Finished, Available, Finished, False)

True

In [61]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 66, Finished, Available, Finished, False)

<Spark SQL result set with 23 rows and 2 fields>

In [62]:
deltastream.stop()

StatementMeta(, 2e23f9ad-7554-49e0-a85f-a7713d5d6da7, 67, Finished, Available, Finished, False)